# Stock Risk KG Agent — starter notebook

Guided walkthrough of the pipeline: **Ingest -> Graph -> Risk -> Query -> Orchestrate**.
Fill in each `# TODO` — the corresponding module in `src/` already has a working
reference implementation if you get stuck; this notebook is meant to build your
own mental model of *why* each step exists, not to hide the logic from you.

Before running: copy `.env.example` to `.env`, fill in `NEO4J_*` and `LLM_API_KEY`,
and drop `prices_<TICKER>.csv` files into `data/raw/` (see `data/raw/README.md`).

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))

## Part 1 — Ground: turn raw CSVs into a clean returns matrix

This is the foundation of "Ground": every later claim about risk has to trace
back to this data, not to the LLM's memory of what AAPL's volatility "usually" is.

In [ ]:
from src.ingest import fetch_investing, clean_returns

# TODO: load and inspect the raw price CSVs
raw_prices = ...  # fetch_investing.load_all_prices()
raw_prices.head()

In [ ]:
# TODO: clean the prices and compute the daily returns matrix
# clean_returns.main() writes data/processed/{prices_long,returns_matrix}.parquet
...

## Part 2 — Build the knowledge graph

Schema first (constraints/indexes), then nodes+edges, then the vector index used
for fuzzy grounding (e.g. "chipmakers" instead of a ticker).

In [ ]:
from src.graph import schema, load_graph, build_correlations, vector_index

# TODO: create constraints/indexes
...

# TODO: load Stock/Sector/PricePoint nodes
...

# TODO: compute CORRELATES_WITH edges from the returns matrix
...

# TODO: embed stocks and build the vector index
...

## Part 3 — Risk scoring + provenance

Composite score from volatility/beta/drawdown/Sharpe, written as `:RiskScore`
nodes with `:DERIVED_FROM`/`:COMPUTED_BY` provenance attached.

In [ ]:
from src.risk import risk_score

# TODO: compute and write risk scores (this also attaches provenance)
...

## Part 4 — Query: natural language -> Cypher

The LLM's only job here is translation. It never answers from memory.

In [ ]:
from src.query import text2cypher

# TODO: ask a question and inspect the generated Cypher + results
question = "Which stocks have the highest risk score right now?"
answer = ...  # text2cypher.ask(question)
answer

## Part 5 — Orchestration: Ground -> Query -> Audit

Put it all together. The pipeline should ground any tickers/sectors mentioned,
run the query, and attach an audit trail for any RiskScore in the results.

In [ ]:
from src.agent import risk_agent

# TODO: run a question through the full pipeline and print the formatted answer
question = "How risky is AAPL compared to other stocks in its sector?"
...